# Manual Prompt Testing

Einfaches Notebook um Prompts Step-by-Step zu testen.
Jeden Cell einzeln ausführen und Output anschauen.

## Setup

In [ ]:
import os
import json
from pathlib import Path
from anthropic import Anthropic

# Wechsel ins Parent-Verzeichnis (Trading_Harry)
os.chdir('..')

# Initialize client
client = Anthropic()
print("✅ Anthropic Client initialized")
print(f"📁 Working directory: {os.getcwd()}")

## Schritt 1: Prompt auswählen

Wähle welchen Prompt du testen möchtest:

In [ ]:
# Verfügbare Prompts (aktive Versionen)
prompts = {
    "deep_analysis": "prompts/deep_analysis_v2.txt",
    "commodities_crypto": "prompts/commodities_crypto_v3.txt",
    "broad_scan": "prompts/broad_scan_v1.txt",
    "portfolio_check": "prompts/portfolio_check_v2.txt",
    "market_context": "prompts/market_context_v1.txt",
    "trend_analyzer": "prompts/trend_analyzer_v1.txt",
    "trade_proposals": "prompts/trade_proposals_v1.txt",
    "policy_monitor": "prompts/policy_monitor_v1.txt",
}

print("Verfügbare Prompts:")
for name, path in prompts.items():
    print(f"  • {name:20} → {path}")

In [ ]:
# 👇 HIER ANPASSEN: Welchen Prompt willst du testen?
SELECTED_PROMPT = "deep_analysis"  # Ändern zu: commodities_crypto, broad_scan, etc.

prompt_path = Path(".").parent / prompts[SELECTED_PROMPT]

if not prompt_path.exists():
    print(f"❌ Datei nicht gefunden: {prompt_path}")
else:
    prompt_text = prompt_path.read_text()
    print(f"✅ Prompt geladen: {SELECTED_PROMPT}")
    print(f"📄 Dateigröße: {len(prompt_text)} Zeichen")
    print(f"\n--- Prompt-Vorschau (erste 500 Zeichen) ---")
    print(prompt_text[:500])
    print("...")

## Schritt 2: Test-Eingabe vorbereiten

Schreib hier deine Test-Daten rein:

In [ ]:
# 👇 HIER ANPASSEN: Deine Test-Eingabe
# Beispiel für deep_analysis: ein Ticker mit Daten

test_input = """{
  "ticker": "AAPL",
  "price": 210.50,
  "sector": "Technology",
  "quick_filter_score": 7.5,
  "technical_signal": "bullish"
}"""

print("Test-Eingabe:")
print(test_input)

## Schritt 3: Claude aufrufen

In [ ]:
print(f"🚀 Rufe Claude auf mit Prompt: {SELECTED_PROMPT}\n")

try:
    response = client.messages.create(
        model="claude-sonnet-5",  # Claude 5 Standard (schneller & billiger)
        max_tokens=2000,
        system=prompt_text,  # Der Prompt als System-Nachricht
        messages=[
            {
                "role": "user",
                "content": test_input  # Die Test-Eingabe
            }
        ]
    )
    
    print("✅ Response erhalten!\n")
    
except Exception as e:
    print(f"❌ Fehler: {e}")
    response = None

## Schritt 4: Response anschauen

In [ ]:
if response:
    # Text-Output
    text = response.content[0].text
    print("--- Response Text ---\n")
    print(text)

## Schritt 5: Usage & Kosten

In [ ]:
if response:
    usage = response.usage
    
    print(f"Input tokens:  {usage.input_tokens}")
    print(f"Output tokens: {usage.output_tokens}")
    print(f"Stop reason:   {response.stop_reason}")
    
    # Kostenschätzung (Sonnet 5 — Claude 5 Standard)
    input_cost = usage.input_tokens * (2 / 1_000_000)    # $2 pro 1M input tokens
    output_cost = usage.output_tokens * (10 / 1_000_000)  # $10 pro 1M output tokens
    total_cost = input_cost + output_cost
    
    print(f"\n💰 Geschätzte Kosten (Sonnet 5): ${total_cost:.6f}")

## Optional: JSON-Parsing versuchen

Falls die Response JSON sein soll:

In [ ]:
if response:
    text = response.content[0].text
    
    # Versuche JSON zu extrahieren
    try:
        # Wenn es Markdown-Code-Block ist:
        if "```json" in text:
            json_str = text.split("```json")[1].split("```")[0].strip()
        else:
            json_str = text
        
        parsed = json.loads(json_str)
        print("✅ JSON geparst:")
        print(json.dumps(parsed, indent=2))
    except json.JSONDecodeError as e:
        print(f"❌ Kein gültiges JSON: {e}")
    except Exception as e:
        print(f"⚠️ Fehler beim Parsing: {e}")

## Tips

1. **Prompt ändern:** Bearbeite `SELECTED_PROMPT` in Schritt 1
2. **Test-Input ändern:** Bearbeite `test_input` in Schritt 2
3. **Modell wechseln:** Ändere `model=` in Schritt 3 (z.B. `claude-sonnet-4-20250514` oder `claude-haiku-4-5-20251001`)
4. **Tokens erhöhen:** Ändere `max_tokens=2000` falls die Response abgeschnitten wird
5. **Jeder Cell einzeln ausführen:** Oben auf den ▶️-Button klicken oder Shift+Enter drücken